# 19 — Susceptibility Meta-Model

This notebook tests whether subject-level classification susceptibility can be predicted from EEG signal-quality and ICA/source-structure features.

Motivation:

> Ideally we would develop a meta-model that can predict which subject / recording will be easiest to classify based on raw signal features alone.

The notebook uses previously computed features:

- Raw/ERP/power features from Notebook 17b
- ICA/source-separation features from Notebook 18

Targets:

- LDA within-subject AUC
- Logistic Regression within-subject AUC
- EEGNet within-subject AUC
- Mean AUC across the three models

Feature groups:

- Raw/ERP/power features only
- ICA features only
- Combined raw + ICA features

Evaluation:

- Leave-one-dataset-out validation
- Models are trained on three datasets and tested on the held-out dataset

Because the sample size is small, this notebook uses simple regularized regressors and interprets the meta-model results cautiously.

In [ ]:
# Run from the repository root so that relative result paths resolve
# identically regardless of where this notebook lives.
from oddball_eeg.paths import use_project_root

PROJECT_ROOT = use_project_root()

In [ ]:
from __future__ import annotations

from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.stats import pearsonr, spearmanr

from sklearn.dummy import DummyRegressor
from sklearn.linear_model import RidgeCV, ElasticNetCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from oddball_eeg.io import save_dataframe
from oddball_eeg.stats import safe_corr, zscore_within_group
from oddball_eeg.visualization import (
    round_numeric_columns,
    save_figure,
    set_pandas_display_options,
)

set_pandas_display_options()

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)

# ------------------------------------------------------------
# Inputs from Notebook 17b and Notebook 18
# ------------------------------------------------------------

RAW_TABLE_DIR = Path("results") / "thesis_artifacts" / "multimodel_classification_susceptibility" / "tables"
ICA_TABLE_DIR = Path("results") / "thesis_artifacts" / "ica_multimodel_classification_susceptibility" / "tables"

RAW_MULTIMODEL_PATH = RAW_TABLE_DIR / "multimodel_subject_susceptibility_features.csv"
ICA_FEATURE_PATH = ICA_TABLE_DIR / "ica_subject_features.csv"

# ------------------------------------------------------------
# Output
# ------------------------------------------------------------

OUT_ROOT = Path("results") / "thesis_artifacts" / "susceptibility_meta_model"
TABLE_DIR = OUT_ROOT / "tables"
FIG_DIR = OUT_ROOT / "figures"

TABLE_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# Models and targets
# ------------------------------------------------------------

MODEL_NAMES = ["lda", "logreg", "eegnet"]

MODEL_LABELS = {
    "lda": "LDA",
    "logreg": "Logistic Regression",
    "eegnet": "EEGNet",
}

RANDOM_STATE = 97

print("Raw multimodel input:", RAW_MULTIMODEL_PATH)
print("ICA feature input:", ICA_FEATURE_PATH)
print("Output root:", OUT_ROOT)

In [ ]:
if not RAW_MULTIMODEL_PATH.exists():
    raise FileNotFoundError(
        f"Missing Notebook 17b output:\n{RAW_MULTIMODEL_PATH}"
    )

if not ICA_FEATURE_PATH.exists():
    raise FileNotFoundError(
        f"Missing Notebook 18 output:\n{ICA_FEATURE_PATH}"
    )

raw_model_df = pd.read_csv(RAW_MULTIMODEL_PATH)
ica_feature_df = pd.read_csv(ICA_FEATURE_PATH)

# Normalize subject IDs as strings
raw_model_df["subject"] = raw_model_df["subject"].astype(str)
ica_feature_df["subject"] = ica_feature_df["subject"].astype(str)

# Replace infinities with NaN
raw_model_df = raw_model_df.replace([np.inf, -np.inf], np.nan)
ica_feature_df = ica_feature_df.replace([np.inf, -np.inf], np.nan)

print("Raw multimodel rows:", len(raw_model_df))
print("ICA feature rows:", len(ica_feature_df))

display(round_numeric_columns(raw_model_df.head(), 4))
display(round_numeric_columns(ica_feature_df.head(), 4))

In [ ]:
MERGE_KEYS = [
    "experiment_name",
    "dataset",
    "task",
    "event_preset",
    "dataset_label",
    "subject",
]

model_specific_df = raw_model_df.merge(
    ica_feature_df,
    on=MERGE_KEYS,
    how="inner",
    suffixes=("", "_ica_duplicate"),
)

save_dataframe(
    model_specific_df,
    TABLE_DIR / "model_specific_raw_ica_subject_features.csv",
)

print("Merged model-specific rows:", len(model_specific_df))
print("Expected rows from raw multimodel table:", len(raw_model_df))

print("\nRows per dataset/model:")
display(
    model_specific_df
    .groupby(["dataset_label", "model"])["subject"]
    .nunique()
    .unstack()
)

display(round_numeric_columns(model_specific_df.head(), 4))

In [ ]:
# One row per subject/recording, with model AUCs as columns.
auc_wide_df = raw_model_df.pivot_table(
    index=MERGE_KEYS,
    columns="model",
    values="auc",
    aggfunc="mean",
).reset_index()

auc_wide_df.columns.name = None

for model in MODEL_NAMES:
    if model not in auc_wide_df.columns:
        auc_wide_df[model] = np.nan

auc_wide_df["mean_auc_across_models"] = auc_wide_df[MODEL_NAMES].mean(axis=1)
auc_wide_df["std_auc_across_models"] = auc_wide_df[MODEL_NAMES].std(axis=1)
auc_wide_df["max_auc_across_models"] = auc_wide_df[MODEL_NAMES].max(axis=1)
auc_wide_df["min_auc_across_models"] = auc_wide_df[MODEL_NAMES].min(axis=1)

# Raw features are repeated once per model in raw_model_df.
# Keep one row per subject.
raw_subject_feature_df = (
    raw_model_df
    .drop(columns=["model", "model_label", "auc", "auc_seed_std", "pr_auc", "bal_acc", "acc", "f1", "summary_path"], errors="ignore")
    .drop_duplicates(subset=MERGE_KEYS)
    .reset_index(drop=True)
)

general_df = (
    auc_wide_df
    .merge(raw_subject_feature_df, on=MERGE_KEYS, how="inner")
    .merge(ica_feature_df, on=MERGE_KEYS, how="inner", suffixes=("", "_ica_duplicate"))
)

save_dataframe(
    general_df,
    TABLE_DIR / "general_classifiability_raw_ica_subject_features.csv",
)

print("General classifiability rows:", len(general_df))
display(round_numeric_columns(general_df.head(), 4))

In [ ]:
def rmse(y_true, y_pred):
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))


def clean_feature_list(df: pd.DataFrame, feature_cols: list[str], min_nonnull: int = 10) -> list[str]:
    """
    Remove features that are missing, all-NaN, near-constant, or unavailable.
    """
    cleaned = []

    for col in feature_cols:
        if col not in df.columns:
            continue

        x = pd.to_numeric(df[col], errors="coerce").replace([np.inf, -np.inf], np.nan)

        if x.notna().sum() < min_nonnull:
            continue

        if x.nunique(dropna=True) <= 1:
            continue

        cleaned.append(col)

    return cleaned


def sanitize_numeric_columns(df: pd.DataFrame, cols: list[str]) -> pd.DataFrame:
    out = df.copy()

    for col in cols:
        if col in out.columns:
            out[col] = pd.to_numeric(out[col], errors="coerce")
            out[col] = out[col].replace([np.inf, -np.inf], np.nan)

    return out

In [ ]:
RAW_ID_COLS = {
    "experiment_name",
    "dataset",
    "task",
    "event_preset",
    "dataset_label",
    "subject",
    "model",
    "model_label",
    "summary_path",
    "roi_channels",
    "roi_method",
}

RAW_TARGET_COLS = {
    "auc",
    "auc_seed_std",
    "pr_auc",
    "bal_acc",
    "acc",
    "f1",
}

ICA_ID_COLS = {
    "experiment_name",
    "dataset",
    "task",
    "event_preset",
    "dataset_label",
    "subject",
    "montage_name",
}

# Raw numeric features from Notebook 17b
raw_all_features = [
    col for col in raw_model_df.select_dtypes(include=[np.number]).columns
    if col not in RAW_ID_COLS
       and col not in RAW_TARGET_COLS
       and not (col.startswith("power_") and not col.startswith("log_"))
]

# ICA numeric features from Notebook 18
ica_all_features = [
    col for col in ica_feature_df.select_dtypes(include=[np.number]).columns
    if col not in ICA_ID_COLS
]

# Compact core feature sets based on the previous correlation results.
raw_core_features = [
    "baseline_rms_uv",
    "baseline_std_uv",
    "roi_baseline_std_uv",
    "roi_p300_peak_signed_uv",
    "roi_p300_peak_abs_uv",
    "roi_p300_rms_diff_uv",
    "roi_erp_snr_db",
    "trial_roi_p300_cohen_d",
    "trial_roi_p300_abs_cohen_d",
    "log_power_theta_4_8",
    "log_power_beta_13_30",
    "log_power_gamma_30_45",
    "log_low_high_ratio",
    "log_gamma_low_ratio",
]

ica_core_features = [
    "mean_ic_spectral_entropy",
    "median_ic_spectral_entropy",
    "min_ic_spectral_entropy",
    "mean_abs_ic_kurtosis",
    "median_abs_ic_kurtosis",
    "max_abs_ic_kurtosis",
    "mean_abs_ic_corr",
    "median_abs_ic_corr",
    "mean_pairwise_ic_mi_approx",
    "median_pairwise_ic_mi_approx",
    "mean_ic_spatial_roughness",
    "median_ic_spatial_roughness",
    "mean_ic_spatial_smoothness",
    "mean_component_map_sparsity",
    "median_component_map_sparsity",
    "mixing_condition_number",
    "unmixing_condition_number",
    "mean_abs_mixing_weight",
    "mean_abs_unmixing_weight",
]

raw_core_features = clean_feature_list(model_specific_df, raw_core_features)
ica_core_features = clean_feature_list(model_specific_df, ica_core_features)
raw_all_features = clean_feature_list(model_specific_df, raw_all_features)
ica_all_features = clean_feature_list(model_specific_df, ica_all_features)

feature_sets = {
    "raw_core": raw_core_features,
    "ica_core": ica_core_features,
    "combined_core": raw_core_features + ica_core_features,
    "raw_all": raw_all_features,
    "ica_all": ica_all_features,
    "combined_all": raw_all_features + ica_all_features,
}

feature_set_summary_df = pd.DataFrame([
    {
        "feature_set": name,
        "n_features": len(cols),
        "features": ",".join(cols),
    }
    for name, cols in feature_sets.items()
])

save_dataframe(
    feature_set_summary_df,
    TABLE_DIR / "feature_set_summary.csv",
)

display(feature_set_summary_df)

In [ ]:
# Model-specific z-scoring
model_specific_z_df = model_specific_df.copy()

model_specific_z_df["auc_z"] = (
    model_specific_z_df
    .groupby(["model", "dataset_label"])["auc"]
    .transform(zscore_within_group)
)

for feature in sorted(set(raw_all_features + ica_all_features)):
    if feature in model_specific_z_df.columns:
        model_specific_z_df[f"{feature}_z"] = (
            model_specific_z_df
            .groupby(["model", "dataset_label"])[feature]
            .transform(zscore_within_group)
        )

# General classifiability z-scoring
general_z_df = general_df.copy()

general_z_df["mean_auc_z"] = (
    general_z_df
    .groupby("dataset_label")["mean_auc_across_models"]
    .transform(zscore_within_group)
)

for feature in sorted(set(raw_all_features + ica_all_features)):
    if feature in general_z_df.columns:
        general_z_df[f"{feature}_z"] = (
            general_z_df
            .groupby("dataset_label")[feature]
            .transform(zscore_within_group)
        )

save_dataframe(
    model_specific_z_df,
    TABLE_DIR / "model_specific_raw_ica_subject_features_zscored.csv",
)

save_dataframe(
    general_z_df,
    TABLE_DIR / "general_classifiability_raw_ica_subject_features_zscored.csv",
)

print("Model-specific z-scored rows:", len(model_specific_z_df))
print("General z-scored rows:", len(general_z_df))

In [ ]:
def make_meta_models():
    """
    Simple meta-models appropriate for small subject-level data.

    DummyMean is included as a baseline.
    Ridge and ElasticNet are the main interpretable regularized models.
    """
    return {
        "dummy_mean": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("model", DummyRegressor(strategy="mean")),
        ]),

        "ridge": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("model", RidgeCV(
                alphas=np.logspace(-3, 3, 25),
            )),
        ]),

        "elasticnet": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("model", ElasticNetCV(
                l1_ratio=[0.05, 0.1, 0.25, 0.5, 0.75, 0.9],
                alphas=np.logspace(-3, 2, 30),
                cv=3,
                random_state=RANDOM_STATE,
                max_iter=50000,
            )),
        ]),
    }


def compute_prediction_metrics(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)

    mask = np.isfinite(y_true) & np.isfinite(y_pred)

    y_true = y_true[mask]
    y_pred = y_pred[mask]

    if len(y_true) < 3:
        return {
            "n": len(y_true),
            "mae": np.nan,
            "rmse": np.nan,
            "r2": np.nan,
            "pearson_r": np.nan,
            "pearson_p": np.nan,
            "spearman_rho": np.nan,
            "spearman_p": np.nan,
        }

    pearson_r, pearson_p, _ = safe_corr(y_true, y_pred, method="pearson")
    spearman_rho, spearman_p, _ = safe_corr(y_true, y_pred, method="spearman")

    try:
        r2 = float(r2_score(y_true, y_pred))
    except Exception:
        r2 = np.nan

    return {
        "n": int(len(y_true)),
        "mae": float(mean_absolute_error(y_true, y_pred)),
        "rmse": rmse(y_true, y_pred),
        "r2": r2,
        "pearson_r": pearson_r,
        "pearson_p": pearson_p,
        "spearman_rho": spearman_rho,
        "spearman_p": spearman_p,
    }

In [ ]:
def evaluate_leave_one_dataset_out(
        *,
        df: pd.DataFrame,
        target_col: str,
        feature_cols: list[str],
        feature_set_name: str,
        target_name: str,
        model_name: str | None = None,
        target_mode: str,
):
    """
    Evaluate meta-models using leave-one-dataset-out validation.

    Each fold:
    - train on all datasets except one
    - test on the held-out dataset
    """
    needed_cols = ["dataset_label", "subject", target_col] + feature_cols
    if model_name is not None:
        needed_cols.append("model")

    work_df = df[needed_cols].copy()
    work_df = sanitize_numeric_columns(work_df, [target_col] + feature_cols)

    # Keep rows with finite target
    work_df = work_df[np.isfinite(work_df[target_col])].copy()

    # Remove features that are unusable in this dataframe.
    usable_features = clean_feature_list(work_df, feature_cols, min_nonnull=10)

    if len(usable_features) == 0:
        return pd.DataFrame(), pd.DataFrame()

    X = work_df[usable_features]
    y = work_df[target_col].astype(float)
    groups = work_df["dataset_label"].astype(str)

    logo = LeaveOneGroupOut()
    meta_models = make_meta_models()

    pred_rows = []
    fold_rows = []

    for fold_idx, (train_idx, test_idx) in enumerate(logo.split(X, y, groups)):
        train_groups = sorted(groups.iloc[train_idx].unique())
        test_group = str(groups.iloc[test_idx].iloc[0])

        X_train = X.iloc[train_idx]
        X_test = X.iloc[test_idx]
        y_train = y.iloc[train_idx]
        y_test = y.iloc[test_idx]

        for meta_model_name, estimator in meta_models.items():
            try:
                estimator.fit(X_train, y_train)
                y_pred = estimator.predict(X_test)

                fold_metrics = compute_prediction_metrics(y_test, y_pred)

                fold_rows.append({
                    "target_name": target_name,
                    "target_mode": target_mode,
                    "classification_model": model_name if model_name is not None else "mean_across_models",
                    "feature_set": feature_set_name,
                    "meta_model": meta_model_name,
                    "fold": fold_idx,
                    "heldout_dataset": test_group,
                    "train_datasets": ",".join(train_groups),
                    "n_train": int(len(train_idx)),
                    "n_test": int(len(test_idx)),
                    "n_features": int(len(usable_features)),
                    **fold_metrics,
                })

                for local_i, global_i in enumerate(test_idx):
                    row = work_df.iloc[global_i]

                    pred_rows.append({
                        "target_name": target_name,
                        "target_mode": target_mode,
                        "classification_model": model_name if model_name is not None else "mean_across_models",
                        "feature_set": feature_set_name,
                        "meta_model": meta_model_name,
                        "fold": fold_idx,
                        "heldout_dataset": test_group,
                        "dataset_label": row["dataset_label"],
                        "subject": row["subject"],
                        "y_true": float(y_test.iloc[local_i]),
                        "y_pred": float(y_pred[local_i]),
                        "n_features": int(len(usable_features)),
                    })

            except Exception as e:
                fold_rows.append({
                    "target_name": target_name,
                    "target_mode": target_mode,
                    "classification_model": model_name if model_name is not None else "mean_across_models",
                    "feature_set": feature_set_name,
                    "meta_model": meta_model_name,
                    "fold": fold_idx,
                    "heldout_dataset": test_group,
                    "train_datasets": ",".join(train_groups),
                    "n_train": int(len(train_idx)),
                    "n_test": int(len(test_idx)),
                    "n_features": int(len(usable_features)),
                    "error": f"{type(e).__name__}: {e}",
                })

    pred_df = pd.DataFrame(pred_rows)
    fold_df = pd.DataFrame(fold_rows)

    return pred_df, fold_df


def summarize_predictions(pred_df: pd.DataFrame) -> pd.DataFrame:
    rows = []

    group_cols = [
        "target_name",
        "target_mode",
        "classification_model",
        "feature_set",
        "meta_model",
    ]

    for keys, g in pred_df.groupby(group_cols):
        metrics = compute_prediction_metrics(g["y_true"], g["y_pred"])

        rows.append({
            **dict(zip(group_cols, keys)),
            "n_features": int(g["n_features"].iloc[0]),
            **metrics,
        })

    out = pd.DataFrame(rows)

    if len(out) == 0:
        return out

    out = out.sort_values(
        ["target_mode", "classification_model", "spearman_rho", "rmse"],
        ascending=[True, True, False, True],
    ).reset_index(drop=True)

    return out

In [ ]:
all_pred_frames = []
all_fold_frames = []

# ------------------------------------------------------------
# Primary analysis:
# Predict within-dataset z-scored AUC.
# This asks: within each dataset, can we predict which subjects are easier?
# ------------------------------------------------------------

for classification_model in MODEL_NAMES:
    df_model = model_specific_z_df[
        model_specific_z_df["model"] == classification_model
        ].copy()

    for feature_set_name, feature_cols_original in feature_sets.items():
        z_feature_cols = [
            f"{col}_z"
            for col in feature_cols_original
            if f"{col}_z" in df_model.columns
        ]

        pred_df, fold_df = evaluate_leave_one_dataset_out(
            df=df_model,
            target_col="auc_z",
            feature_cols=z_feature_cols,
            feature_set_name=feature_set_name,
            target_name=f"{MODEL_LABELS[classification_model]} AUC",
            model_name=classification_model,
            target_mode="within_dataset_z",
        )

        all_pred_frames.append(pred_df)
        all_fold_frames.append(fold_df)

# ------------------------------------------------------------
# Secondary analysis:
# Predict raw AUC directly.
# This is harder because dataset-level differences remain.
# ------------------------------------------------------------

for classification_model in MODEL_NAMES:
    df_model = model_specific_df[
        model_specific_df["model"] == classification_model
        ].copy()

    for feature_set_name, feature_cols_original in feature_sets.items():
        pred_df, fold_df = evaluate_leave_one_dataset_out(
            df=df_model,
            target_col="auc",
            feature_cols=feature_cols_original,
            feature_set_name=feature_set_name,
            target_name=f"{MODEL_LABELS[classification_model]} AUC",
            model_name=classification_model,
            target_mode="raw_auc",
        )

        all_pred_frames.append(pred_df)
        all_fold_frames.append(fold_df)

model_specific_pred_df = pd.concat(
    [df for df in all_pred_frames if df is not None and len(df) > 0],
    ignore_index=True,
)

model_specific_fold_df = pd.concat(
    [df for df in all_fold_frames if df is not None and len(df) > 0],
    ignore_index=True,
)

model_specific_summary_df = summarize_predictions(model_specific_pred_df)

save_dataframe(
    model_specific_pred_df,
    TABLE_DIR / "model_specific_lodo_predictions.csv",
)

save_dataframe(
    model_specific_fold_df,
    TABLE_DIR / "model_specific_lodo_fold_metrics.csv",
)

save_dataframe(
    model_specific_summary_df,
    TABLE_DIR / "model_specific_lodo_summary_metrics.csv",
)

display(round_numeric_columns(model_specific_summary_df.head(40), 4))

In [ ]:
general_pred_frames = []
general_fold_frames = []

# Primary: within-dataset z-scored mean AUC
for feature_set_name, feature_cols_original in feature_sets.items():
    z_feature_cols = [
        f"{col}_z"
        for col in feature_cols_original
        if f"{col}_z" in general_z_df.columns
    ]

    pred_df, fold_df = evaluate_leave_one_dataset_out(
        df=general_z_df,
        target_col="mean_auc_z",
        feature_cols=z_feature_cols,
        feature_set_name=feature_set_name,
        target_name="Mean AUC across models",
        model_name=None,
        target_mode="within_dataset_z",
    )

    general_pred_frames.append(pred_df)
    general_fold_frames.append(fold_df)

# Secondary: raw mean AUC
for feature_set_name, feature_cols_original in feature_sets.items():
    pred_df, fold_df = evaluate_leave_one_dataset_out(
        df=general_df,
        target_col="mean_auc_across_models",
        feature_cols=feature_cols_original,
        feature_set_name=feature_set_name,
        target_name="Mean AUC across models",
        model_name=None,
        target_mode="raw_auc",
    )

    general_pred_frames.append(pred_df)
    general_fold_frames.append(fold_df)

general_pred_df = pd.concat(
    [df for df in general_pred_frames if df is not None and len(df) > 0],
    ignore_index=True,
)

general_fold_df = pd.concat(
    [df for df in general_fold_frames if df is not None and len(df) > 0],
    ignore_index=True,
)

general_summary_df = summarize_predictions(general_pred_df)

save_dataframe(
    general_pred_df,
    TABLE_DIR / "general_classifiability_lodo_predictions.csv",
)

save_dataframe(
    general_fold_df,
    TABLE_DIR / "general_classifiability_lodo_fold_metrics.csv",
)

save_dataframe(
    general_summary_df,
    TABLE_DIR / "general_classifiability_lodo_summary_metrics.csv",
)

display(round_numeric_columns(general_summary_df.head(40), 4))

In [ ]:
all_summary_df = pd.concat(
    [
        model_specific_summary_df,
        general_summary_df,
    ],
    ignore_index=True,
)

# Add improvement over dummy within each target/mode/classification_model/feature_set.
dummy_ref = (
    all_summary_df[all_summary_df["meta_model"] == "dummy_mean"]
    [[
        "target_name",
        "target_mode",
        "classification_model",
        "feature_set",
        "rmse",
        "mae",
        "spearman_rho",
    ]]
    .rename(columns={
        "rmse": "dummy_rmse",
        "mae": "dummy_mae",
        "spearman_rho": "dummy_spearman_rho",
    })
)

all_summary_df = all_summary_df.merge(
    dummy_ref,
    on=[
        "target_name",
        "target_mode",
        "classification_model",
        "feature_set",
    ],
    how="left",
)

all_summary_df["rmse_improvement_vs_dummy"] = (
        all_summary_df["dummy_rmse"] - all_summary_df["rmse"]
)

all_summary_df["mae_improvement_vs_dummy"] = (
        all_summary_df["dummy_mae"] - all_summary_df["mae"]
)

all_summary_df["rmse_improvement_pct_vs_dummy"] = (
        100.0 * all_summary_df["rmse_improvement_vs_dummy"] / all_summary_df["dummy_rmse"]
)

ranked_summary_df = all_summary_df.sort_values(
    [
        "target_mode",
        "classification_model",
        "spearman_rho",
        "rmse_improvement_pct_vs_dummy",
    ],
    ascending=[True, True, False, False],
).reset_index(drop=True)

save_dataframe(
    all_summary_df,
    TABLE_DIR / "all_lodo_summary_metrics.csv",
)

save_dataframe(
    ranked_summary_df,
    TABLE_DIR / "all_lodo_summary_metrics_ranked.csv",
)

display(round_numeric_columns(ranked_summary_df.head(60), 4))

In [ ]:
# Prefer non-dummy models for interpretation.
non_dummy = ranked_summary_df[
    ranked_summary_df["meta_model"] != "dummy_mean"
    ].copy()

best_by_target_df = (
    non_dummy
    .sort_values(
        [
            "target_mode",
            "classification_model",
            "spearman_rho",
            "rmse_improvement_pct_vs_dummy",
        ],
        ascending=[True, True, False, False],
    )
    .groupby(["target_mode", "classification_model"])
    .head(5)
    .reset_index(drop=True)
)

save_dataframe(
    best_by_target_df,
    TABLE_DIR / "best_meta_models_by_target.csv",
)

display(round_numeric_columns(best_by_target_df, 4))

In [ ]:
plot_df = ranked_summary_df[
    (ranked_summary_df["target_mode"] == "within_dataset_z")
    & (ranked_summary_df["meta_model"] != "dummy_mean")
    ].copy()

# Use best score per classification target / feature set / meta-model
plot_df["label"] = (
        plot_df["classification_model"].astype(str)
        + "\n"
        + plot_df["feature_set"].astype(str)
        + "\n"
        + plot_df["meta_model"].astype(str)
)

# Keep top 25 by Spearman
plot_df = plot_df.sort_values("spearman_rho", ascending=False).head(25)
plot_df = plot_df.sort_values("spearman_rho", ascending=True)

fig, ax = plt.subplots(figsize=(10, 9))

ax.barh(
    plot_df["label"],
    plot_df["spearman_rho"],
)

ax.axvline(0.0, linewidth=1)
ax.set_xlabel("LODO Spearman correlation: predicted vs true z-scored susceptibility")
ax.set_title("Best leave-one-dataset-out meta-model results")
ax.grid(axis="x", alpha=0.3)

fig.tight_layout()
save_figure(fig, FIG_DIR / "best_lodo_meta_model_spearman_barplot.png")
plt.show()

In [ ]:
plot_df = ranked_summary_df[
    (ranked_summary_df["target_mode"] == "within_dataset_z")
    & (ranked_summary_df["meta_model"] != "dummy_mean")
    ].copy()

plot_df["label"] = (
        plot_df["classification_model"].astype(str)
        + "\n"
        + plot_df["feature_set"].astype(str)
        + "\n"
        + plot_df["meta_model"].astype(str)
)

plot_df = plot_df.sort_values("rmse_improvement_pct_vs_dummy", ascending=False).head(25)
plot_df = plot_df.sort_values("rmse_improvement_pct_vs_dummy", ascending=True)

fig, ax = plt.subplots(figsize=(10, 9))

ax.barh(
    plot_df["label"],
    plot_df["rmse_improvement_pct_vs_dummy"],
)

ax.axvline(0.0, linewidth=1)
ax.set_xlabel("RMSE improvement over dummy mean baseline (%)")
ax.set_title("Leave-one-dataset-out meta-model improvement over baseline")
ax.grid(axis="x", alpha=0.3)

fig.tight_layout()
save_figure(fig, FIG_DIR / "best_lodo_meta_model_rmse_improvement_barplot.png")
plt.show()

In [ ]:
combined_pred_df = pd.concat(
    [
        model_specific_pred_df,
        general_pred_df,
    ],
    ignore_index=True,
)

# Select one best usable model per target for z-scored susceptibility.
# A model must improve over the dummy baseline in RMSE.
best_candidates_df = non_dummy[
    (non_dummy["target_mode"] == "within_dataset_z")
    & (non_dummy["rmse_improvement_pct_vs_dummy"] > 0)
    ].copy()

best_one_per_target_df = (
    best_candidates_df
    .sort_values(
        ["classification_model", "spearman_rho", "rmse_improvement_pct_vs_dummy"],
        ascending=[True, False, False],
    )
    .groupby("classification_model")
    .head(1)
    .reset_index(drop=True)
)

save_dataframe(
    best_one_per_target_df,
    TABLE_DIR / "best_single_meta_model_per_target.csv",
)

display(round_numeric_columns(best_one_per_target_df, 4))

for _, best in best_one_per_target_df.iterrows():
    classification_model = best["classification_model"]
    feature_set = best["feature_set"]
    meta_model = best["meta_model"]
    target_mode = best["target_mode"]

    g = combined_pred_df[
        (combined_pred_df["classification_model"] == classification_model)
        & (combined_pred_df["feature_set"] == feature_set)
        & (combined_pred_df["meta_model"] == meta_model)
        & (combined_pred_df["target_mode"] == target_mode)
        ].copy()

    if len(g) == 0:
        continue

    fig, ax = plt.subplots(figsize=(7, 5))

    for dataset_label, sub in g.groupby("dataset_label"):
        ax.scatter(
            sub["y_true"],
            sub["y_pred"],
            label=dataset_label,
            alpha=0.8,
        )

    lim_min = np.nanmin([g["y_true"].min(), g["y_pred"].min()])
    lim_max = np.nanmax([g["y_true"].max(), g["y_pred"].max()])

    ax.plot(
        [lim_min, lim_max],
        [lim_min, lim_max],
        linestyle="--",
        linewidth=1,
    )

    ax.set_xlabel("True susceptibility")
    ax.set_ylabel("Predicted susceptibility")
    ax.set_title(
        f"{classification_model}: {feature_set} + {meta_model}\n"
        f"Spearman={best['spearman_rho']:.3f}, RMSE={best['rmse']:.3f}"
    )
    ax.grid(True, alpha=0.3)
    ax.legend()

    fig.tight_layout()
    save_figure(
        fig,
        FIG_DIR / f"best_meta_model_scatter_{classification_model}_{feature_set}_{meta_model}.png",
    )
    plt.show()

In [ ]:
def fit_final_linear_model_and_coefficients(
        *,
        df: pd.DataFrame,
        target_col: str,
        feature_cols: list[str],
        meta_model_name: str,
):
    """
    Fit final model on all available rows and extract standardized coefficients.
    Works for RidgeCV and ElasticNetCV.
    """
    usable_features = clean_feature_list(df, feature_cols, min_nonnull=10)

    if len(usable_features) == 0:
        return pd.DataFrame()

    work_df = df[usable_features + [target_col]].copy()
    work_df = sanitize_numeric_columns(work_df, usable_features + [target_col])
    work_df = work_df[np.isfinite(work_df[target_col])].copy()

    if len(work_df) < 5:
        return pd.DataFrame()

    X = work_df[usable_features]
    y = work_df[target_col].astype(float)

    models = make_meta_models()

    if meta_model_name not in models:
        return pd.DataFrame()

    estimator = models[meta_model_name]
    estimator.fit(X, y)

    final_model = estimator.named_steps["model"]

    if not hasattr(final_model, "coef_"):
        return pd.DataFrame()

    coefs = np.asarray(final_model.coef_, dtype=float).ravel()

    coef_df = pd.DataFrame({
        "feature": usable_features,
        "coefficient": coefs,
        "abs_coefficient": np.abs(coefs),
    })

    coef_df = coef_df.sort_values(
        "abs_coefficient",
        ascending=False,
    ).reset_index(drop=True)

    return coef_df


coef_frames = []

for _, best in best_one_per_target_df.iterrows():
    classification_model = best["classification_model"]
    feature_set = best["feature_set"]
    meta_model_name = best["meta_model"]
    target_mode = best["target_mode"]

    if meta_model_name == "dummy_mean":
        continue

    if classification_model == "mean_across_models":
        df_source = general_z_df.copy()
        target_col = "mean_auc_z"
    else:
        df_source = model_specific_z_df[
            model_specific_z_df["model"] == classification_model
            ].copy()
        target_col = "auc_z"

    feature_cols_original = feature_sets[feature_set]
    feature_cols_z = [
        f"{col}_z"
        for col in feature_cols_original
        if f"{col}_z" in df_source.columns
    ]

    coef_df = fit_final_linear_model_and_coefficients(
        df=df_source,
        target_col=target_col,
        feature_cols=feature_cols_z,
        meta_model_name=meta_model_name,
    )

    if len(coef_df) == 0:
        continue

    coef_df["classification_model"] = classification_model
    coef_df["feature_set"] = feature_set
    coef_df["meta_model"] = meta_model_name
    coef_df["target_mode"] = target_mode

    # Remove _z suffix for readability
    coef_df["feature_clean"] = coef_df["feature"].str.replace("_z$", "", regex=True)

    coef_frames.append(coef_df)

if len(coef_frames) > 0:
    coefficients_df = pd.concat(coef_frames, ignore_index=True)
else:
    coefficients_df = pd.DataFrame()

save_dataframe(
    coefficients_df,
    TABLE_DIR / "best_meta_model_coefficients.csv",
)

display(round_numeric_columns(coefficients_df.head(50), 4))

In [ ]:
if len(coefficients_df) > 0:
    for (classification_model, feature_set, meta_model), g in coefficients_df.groupby(
            ["classification_model", "feature_set", "meta_model"]
    ):
        plot_df = g.sort_values("abs_coefficient", ascending=False).head(15)
        plot_df = plot_df.sort_values("coefficient", ascending=True)

        fig, ax = plt.subplots(figsize=(9, 6))

        ax.barh(
            plot_df["feature_clean"],
            plot_df["coefficient"],
        )

        ax.axvline(0.0, linewidth=1)
        ax.set_xlabel("Standardized coefficient")
        ax.set_title(
            f"Best meta-model coefficients\n"
            f"{classification_model}: {feature_set} + {meta_model}"
        )
        ax.grid(axis="x", alpha=0.3)

        fig.tight_layout()
        save_figure(
            fig,
            FIG_DIR / f"best_meta_model_coefficients_{classification_model}_{feature_set}_{meta_model}.png",
        )
        plt.show()
else:
    print("No coefficients available to plot.")

In [ ]:
print("=" * 90)
print("Meta-model interpretation helper")
print("=" * 90)

print("\nBest leave-one-dataset-out models for within-dataset z-scored susceptibility:")
display(
    round_numeric_columns(
        best_one_per_target_df[
            [
                "classification_model",
                "feature_set",
                "meta_model",
                "n",
                "n_features",
                "spearman_rho",
                "spearman_p",
                "pearson_r",
                "rmse",
                "mae",
                "r2",
                "rmse_improvement_pct_vs_dummy",
            ]
        ],
        4,
    )
)

print("\nImportant interpretation notes:")
print("- Positive Spearman means the meta-model ranks easier subjects above harder subjects.")
print("- RMSE improvement over dummy shows whether the model improves over predicting the training mean.")
print("- R2 may be unstable or negative because sample size is small and leave-one-dataset-out is difficult.")
print("- The within-dataset z-scored target is best interpreted as relative susceptibility within each dataset.")
print("- Raw-AUC prediction is harder because dataset-level differences are not removed.")

## Interpretation Template

Use the meta-model results cautiously.

The key question is not whether the meta-model perfectly predicts classification AUC. With only 58 subject-level observations across four datasets, perfect prediction is not expected.

The important questions are:

1. Does the model rank high-susceptibility subjects above low-susceptibility subjects?
2. Do raw/ERP features, ICA features, or combined features work better?
3. Are predictions better for LDA/Logistic Regression than for EEGNet?
4. Do the selected coefficients agree with the correlation results?

Possible conclusions:

- If raw/ERP features outperform ICA features:
  > Subject classifiability is more directly explained by signal quality and ERP separability than by the current ICA-quality metrics.

- If ICA features add improvement:
  > Source-structure metrics contain additional information about which subjects are easier to classify.

- If combined raw + ICA features perform best:
  > Classification susceptibility appears to depend on both raw EEG quality and source-structure organization.

- If meta-model performance is weak:
  > The subject-level predictors show meaningful correlations but are not yet sufficient for reliable out-of-dataset prediction.

In [ ]:
print("Outputs saved under:", OUT_ROOT)

print("\nTables:")
for p in sorted(TABLE_DIR.glob("*.csv")):
    print(" -", p)

print("\nFigures:")
for p in sorted(FIG_DIR.glob("*.png")):
    print(" -", p)